In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
team_name="team_lemma"
catalog=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
dbutils.widgets.text("source_path","abfss://raw@schwabdldevsa.dfs.core.windows.net","SOURCE PATH")
dbutils.widgets.text("target_path",f"/Volumes/{catalog}/landing/PWG/")
# run_id=datetime.now().strftime("%Y%m%d_%H%M%S")

In [0]:
spark.sql(f"use catalog {catalog}")

In [0]:
batch_id=dbutils.widgets.get("batch_id")
source_path=dbutils.widgets.get("source_path")
target_path=dbutils.widgets.get("target_path")

In [0]:
#schema of customer domain of files customer Domain

#customermgmt Columns for Batch1
CUSTOMERMGMT_COLS = [
    "ActionType", "ActionTS",
    "C_ID", "C_TAX_ID", "C_GNDR", "C_TIER", "C_DOB",
    "C_L_NAME", "C_F_NAME", "C_M_NAME",
    "C_ADLINE1", "C_ADLINE2", "C_ZIPCODE", "C_CITY", "C_STATE_PROV", "C_CTRY",
    "C_PRIM_EMAIL", "C_ALT_EMAIL",
    "C_CTRY_1", "C_AREA_1", "C_LOCAL_1", "C_EXT_1",
    "C_CTRY_2", "C_AREA_2", "C_LOCAL_2", "C_EXT_2",
    "C_CTRY_3", "C_AREA_3", "C_LOCAL_3", "C_EXT_3",
    "C_LCL_TX_ID", "C_NAT_TX_ID",
    "CA_ID", "CA_TAX_ST", "CA_B_ID", "CA_NAME"
]


# Customer.txt schema for BATCH 2 and 3 . Thsi File Not in Bacth 1
CUSTOMER_B2_B3_COLS=[
    "CDC_FLAG","CDC_DSN", "C_ID", "C_TAX_ID", "C_ST_ID",
    "C_L_NAME", "C_F_NAME", "C_M_NAME", "C_GNDR", "C_TIER", "C_DOB",
    "C_ADLINE1", "C_ADLINE2", "C_ZIPCODE", "C_CITY", "C_STATE_PROV", "C_CTRY",
    "C_CTRY_1", "C_AREA_1", "C_LOCAL_1", "C_EXT_1",
    "C_CTRY_2", "C_AREA_2", "C_LOCAL_2", "C_EXT_2",
    "C_CTRY_3", "C_AREA_3", "C_LOCAL_3", "C_EXT_3",
    "C_EMAIL_1", "C_EMAIL_2",
    "C_LCL_TX_ID", "C_NAT_TX_ID"
]


#prospect.json for all batch
PROSPECT_SELECT = [
    ("AgencyID","p.agency_id"),
    ("FirstName","p.personal.name.first_name"),
    ("LastName","p.personal.name.last_name"),
    ("MiddleInitial","p.personal.name.middle_initial"),
    ("Gender","p.personal.demographics.gender"),
    ("Age","p.personal.demographics.age"),
    ("MaritalStatus","p.personal.demographics.marital_status"),
    ("AddressLine1","p.contact.address.line1"),
    ("AddressLine2", "p.contact.address.line2"),
    ("City","p.contact.address.city"),
    ("State","p.contact.address.state"),
    ("PostalCode","p.contact.address.postal_code"),
    ("Country","p.contact.address.country"),
    ("Phone","p.contact.phone.full_number"),
    ("Income","p.financial.income.annual_income"),
    ("NetWorth", "p.financial.wealth.net_worth"),
    ("CreditRating","p.financial.credit.credit_rating"),
    ("NumberCreditCards","p.financial.credit.number_credit_cards"),
    ("OwnOrRentFlag","p.lifestyle.housing.own_or_rent"),
    ("NumberChildren", "p.lifestyle.family.number_children"),
    ("NumberCars","p.lifestyle.assets.number_cars"),
    ("Employer","p.employment.employer_name"),
]

#watch History Col For BATCH 1 2 3
WATCHHISTORY_B1_COLS  = ["W_C_ID", "W_S_SYMB", "W_DTS", "W_ACTION"]
WATCHHISTORY_B23_COLS = ["CDC_FLAG", "CDC_DSN"] + WATCHHISTORY_B1_COLS

CUSTOMER_DOMAIN_FILES=[
    {
        "file_name":"CustomerMgmt.xml",
        "columns_by_batch":{"1":CUSTOMERMGMT_COLS},
        "type":"xml",
        "landing_name":"customermgmt"
    },
    {     
        "file_name":"Customer.txt",
        "columns_by_batch":{"2":CUSTOMER_B2_B3_COLS,"3":CUSTOMER_B2_B3_COLS},
        "type":"csv_pipe",
        "landing_name":"customer"
    },
    {     
        "file_name":"prospect.json",
        "columns_by_batch":{"1":PROSPECT_SELECT,"2":PROSPECT_SELECT,"3":PROSPECT_SELECT},
        "type":"json",
        "landing_name":"prospect"
    },
    {
        "file_name":"WatchHistory.txt",
        "columns_by_batch":{"1":WATCHHISTORY_B1_COLS,"2":WATCHHISTORY_B23_COLS,"3":WATCHHISTORY_B23_COLS},
        "type":"csv_pipe",
        "landing_name":"watchhistory"
    }
]

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType


def flatten_xml(df, target_cols):

     return df.select(
        col("_ActionType").alias("ActionType"),
        col("_ActionTS").alias("ActionTS"),
        col("Customer._C_ID").alias("C_ID"),
        col("Customer._C_TAX_ID").alias("C_TAX_ID"),
        col("Customer._C_GNDR").alias("C_GNDR"),
        col("Customer._C_TIER").alias("C_TIER"),
        col("Customer._C_DOB").alias("C_DOB"),
        col("Customer.Name.C_L_NAME").alias("C_L_NAME"),
        col("Customer.Name.C_F_NAME").alias("C_F_NAME"),
        col("Customer.Name.C_M_NAME").alias("C_M_NAME"),
        col("Customer.Address.C_ADLINE1").alias("C_ADLINE1"),
        col("Customer.Address.C_ADLINE2").alias("C_ADLINE2"),
        col("Customer.Address.C_ZIPCODE").alias("C_ZIPCODE"),
        col("Customer.Address.C_CITY").alias("C_CITY"),
        col("Customer.Address.C_STATE_PROV").alias("C_STATE_PROV"),
        col("Customer.Address.C_CTRY").alias("C_CTRY"),
        col("Customer.ContactInfo.C_PRIM_EMAIL").alias("C_PRIM_EMAIL"),
        col("Customer.ContactInfo.C_ALT_EMAIL").alias("C_ALT_EMAIL"),
        
        col("Customer.ContactInfo.C_PHONE_1.C_CTRY_CODE").alias("C_CTRY_1"),
        col("Customer.ContactInfo.C_PHONE_1.C_AREA_CODE").alias("C_AREA_1"),
        col("Customer.ContactInfo.C_PHONE_1.C_LOCAL").alias("C_LOCAL_1"),
        col("Customer.ContactInfo.C_PHONE_1.C_EXT").alias("C_EXT_1"),
        
        col("Customer.ContactInfo.C_PHONE_2.C_CTRY_CODE").alias("C_CTRY_2"),
        col("Customer.ContactInfo.C_PHONE_2.C_AREA_CODE").alias("C_AREA_2"),
        col("Customer.ContactInfo.C_PHONE_2.C_LOCAL").alias("C_LOCAL_2"),
        col("Customer.ContactInfo.C_PHONE_2.C_EXT").alias("C_EXT_2"),
        
        col("Customer.ContactInfo.C_PHONE_3.C_CTRY_CODE").alias("C_CTRY_3"),
        col("Customer.ContactInfo.C_PHONE_3.C_AREA_CODE").alias("C_AREA_3"),
        col("Customer.ContactInfo.C_PHONE_3.C_LOCAL").alias("C_LOCAL_3"),
        col("Customer.ContactInfo.C_PHONE_3.C_EXT").alias("C_EXT_3"),
        
        col("Customer.TaxInfo.C_LCL_TX_ID").alias("C_LCL_TX_ID"),
        col("Customer.TaxInfo.C_NAT_TX_ID").alias("C_NAT_TX_ID"),
        col("Customer.Account._CA_ID").alias("CA_ID"),
        col("Customer.Account._CA_TAX_ST").alias("CA_TAX_ST"),
        col("Customer.Account.CA_B_ID").alias("CA_B_ID"),
        col("Customer.Account.CA_NAME").alias("CA_NAME")
    )

In [0]:
def apply_schema(columns):
    schema=StructType([StructField(c, StringType(), True) for c in columns])
    return schema    

In [0]:
def flatten_json(df,target_cols):
    exploded = df.select(
        explode(col("prospect_batch.prospects")).alias("p")
    )
    select_exprs = [
        col(path).alias(name)
        for name, path in target_cols
    ]
    return exploded.select(*select_exprs)

In [0]:
def load_domain(batch_id):
    try:
        print(f"Raw to Landing For Batch {batch_id}")
        for f in CUSTOMER_DOMAIN_FILES:
            if batch_id in f["columns_by_batch"].keys():
                src=f"{source_path}/Batch{batch_id}/{f['file_name']}"

                if f["type"]=="xml":
                    df=spark.read.format("xml").option("rowTag", "TPCDI:Action").load(src)
                    df=flatten_xml(df,f["columns_by_batch"][batch_id])
                    
                elif f["type"]=="csv_pipe":
                    schema=apply_schema(f["columns_by_batch"][batch_id])
                    df=spark.read.option("header","false").option("sep","|").schema(schema).csv(src)
                elif f["type"]=="json":
                    df=spark.read.option("multiLine",'true').json(src)
                    df=flatten_json(df,f["columns_by_batch"][batch_id])
                run_id=datetime.now().strftime("%Y%m%d_%H%M%S")
                df=df.withColumn("_landing_ts",current_timestamp())\
                    .withColumn("_batch",lit(batch_id))\
                    .withColumn("_source_file",lit(f['file_name']))\
                    .withColumn("_run_id",lit(run_id))
                # df.limit(10).display()
                source_count=df.count()
                print(f"Writing File {f['landing_name']}....")
                df.write.mode("overwrite").parquet(f"{target_path}/Batch{batch_id}/{f['landing_name']}")
                print(f"Write successfully with count {df.count()}")
                target_count=spark.read.parquet(f"{target_path}/Batch{batch_id}/{f['landing_name']}").count()
                    
                log_pipeline_recon(
                    spark=spark,
                    run_id=run_id,
                    batch_id=batch_id,
                    domain="CUSTOMER",
                    table_name=f['landing_name'],
                    source_layer="raw",
                    target_layer="landing",
                    source_count=source_count,
                    target_count=target_count
                )
                log_audit_event(
                    spark=spark,
                    run_id=run_id,
                    batch=batch_id,
                    layer="landing",
                    table_name=f['landing_name'],
                    operation="OVERWRITE",
                    rows_affected=target_count
                )
    except Exception as e:
        print(e)
        raise e
                



In [0]:
load_domain(batch_id)